#Libraries

In [ ]:
%reset -f   

In [ ]:
import pandas as pd
import numpy as np
from numpy import matlib
import matplotlib.pyplot as plt
from numpy import linalg as LA
from sklearn import preprocessing
import keras
from keras.layers import Dense, Activation, Flatten, Input
import tensorflow as tf
from keras.layers import Input, Dense,Reshape
from keras.models import Model
from keras import models,layers, initializers
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from tensorflow.keras.models import model_from_json
import plotly.express as px
import math as mt
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter
import scipy
from scipy import signal, ndimage

#Tick to train a particular model

In [ ]:
#@markdown ### Model Params
aligned_m = True #@param {type:"boolean"}
tilted_m = False #@param {type:"boolean"}
pegboard_m = False #@param {type:"boolean"}
helix_m = False #@param {type:"boolean"}

#functions


In [ ]:
def save_model(model, filename):
    #model_yaml = model.to_yaml()
    #with open(filename + ".yaml", "w") as yaml_file:
    #    yaml_file.write(model_yaml)
    model_json = model.to_json()
    with open(filename + ".json", "w") as json_file:
      json_file.write(model_json)
# serialize weights to HDF5
    model.save_weights(filename + ".h5")
    print("Saved model to disk")

#%% load model function

def load_model(filename):
    #yaml_file = open(filename + ".yaml", 'r')
    #loaded_model_yaml = yaml_file.read()
    #yaml_file.close()
    #loaded_model = model_from_yaml(loaded_model_yaml)
    json_file = open(filename + '.json', 'r')
    loaded_model_json = json_file.read()
    json_file.close()
    loaded_model = model_from_json(loaded_model_json)
    # load weights into new model
    loaded_model.load_weights(filename + ".h5")
    print("Loaded model from disk")
    return loaded_model

#Main Code

###Load splined trajectory

In [ ]:
if aligned_m == True:
  data = pd.read_csv('Trajectory_aligned_no_diagonals_2L_Sep.csv', header=None)
  newpos = np.asarray(data)
  traj = newpos
  print(newpos.shape)
  #simply plots entire trajectory (splined)
  fig = px.line_3d(x=traj[:,0], y=traj[:,1], z=traj[:,2])
  fig.update_layout(
      title="Aligned_lattice Trajectory",
      xaxis_title="X",
      yaxis_title="Y",

      legend_title="Legend Title",
      font=dict(
          family="Courier New, monospace",
          size=18,
          color="RebeccaPurple"
      )
  )
  fig.show()

if tilted_m == True:
  data = pd.read_csv(data_fol + 'Trajectory_interpolated_tilted_lattice.csv', header=None)
  newpos=np.asarray(data)
  traj = newpos
  #Rotation matrix
  x_ang = 45
  y_ang = 55
  rot_x = [[1, 0, 0],
  [0, mt.cos(x_ang*mt.pi/180), -mt.sin(x_ang*mt.pi/180)],
  [0, mt.sin(x_ang*mt.pi/180), mt.cos(x_ang*mt.pi/180)]]
  rot_y = [[mt.cos(y_ang*mt.pi/180), 0, mt.sin(y_ang*mt.pi/180)],
  [0, 1,  0],
  [-mt.sin(y_ang*mt.pi/180), 0, mt.cos(y_ang*mt.pi/180)]]
  rotm = np.dot(rot_x, rot_y)
  tilted=np.dot(traj,rotm)
  print(newpos.shape)
  #simply plots entire trajectory (splined)
  fig = px.line_3d(x=tilted[:,0], y=tilted[:,1], z=tilted[:,2])
  fig.update_layout(
      title="Tilted_lattice Trajectory",
      xaxis_title="X",
      yaxis_title="Y",

      legend_title="Legend Title",
      font=dict(
          family="Courier New, monospace",
          size=18,
          color="RebeccaPurple"
      )
  )
  fig.show()

if pegboard_m == True:
  data = pd.read_csv(data_fol + 'Trajectory_interpolated_pegboard.csv', header=None)
  newpos=np.asarray(data)
  traj = newpos
  print(newpos.shape)

  fig=go.Figure()
  fig.add_trace(go.Scatter3d(x=newpos[:,0], y=newpos[:,1], z=newpos[:,2],mode='lines',opacity=1,marker=dict(size=1), ))
  fig.update_layout(
          scene = {
              "xaxis": {"nticks": 4},
              "xaxis": {"nticks": 1},
              "zaxis": {"nticks": 10},
              'camera_eye': {"x": 2, "y": 0, "z": 1.2},
              "aspectratio": {"x": 0.2, "y": 1, "z": 2}

          })

  fig.show()

if helix_m == True:
  data = pd.read_csv(data_fol+ 'Trajectory_fivecoil_down.csv', header=None)
  newpos=np.asarray(data)
  traj = newpos
  print(newpos.shape)
  #simply plots entire trajectory (splined)
  fig = px.line_3d(x=traj[:,0], y=traj[:,1], z=traj[:,2])
  fig.update_layout(
      title="Helix Trajectory",
      xaxis_title="X",
      yaxis_title="Y",

      legend_title="Legend Title",
      font=dict(
          family="Courier New, monospace",
          size=18,
          color="RebeccaPurple"
      )
  )
  fig.show()

## load 2D traj

In [ ]:
import pickle
traj = 'traj_latcheck_30k.pk1'
with open(traj, "rb") as f:
    d = pickle.load(f)
    f.close()
locals().update(d)

x = np.asarray(x)
y = np.asarray(y)
z = np.zeros(len(x))
newpos = np.column_stack((x,y,z))
pos = newpos
print(newpos.shape)

In [ ]:
plt.scatter(newpos[:,0],newpos[:,1])

#Head Direction

In [ ]:
if aligned_m == 1 or pegboard_m == 1 or helix_m == 1:
  n1 = 20 #bins theta    #azimuth
  n2 = 2 #bins phi      #pitch

  dphi=180/n2   #for aligned
  dtheta=360/n1
  L1_neurons=100 #no. of neurons in layer1 (L1) autoencoder middle layer
  phis=np.arange(0,180+1,dphi) #for aligned
  thetas=np.arange(0,360,dtheta)
  angle_pref=np.zeros((n1,n2+1,3))
  #angle_pref=np.zeros((n1,n2,3))

  print(thetas)
  print(phis)
if tilted_m == 1:
  n1=20 #bins theta    #azimuth
  n2=2 #bins phi      #pitch

  dphi = 90/n2 # for tilted
  dtheta=360/n1
  L1_neurons=100 #no. of neurons in layer1 (L1) autoencoder middle layer
  phis=np.arange(45,180,dphi)   #for tilted
  thetas=np.arange(0,360,dtheta)
  angle_pref=np.zeros((n1,n2,3))
  #angle_pref=np.zeros((n1,n2,3))

  print(thetas)
  print(phis)

In [ ]:
# u_i
temp_phis = np.repeat(phis, len(thetas))
temp_thetas = np.matlib.repmat(thetas, 1,len(phis))
temp_phis = np.transpose(np.asarray(temp_phis))
pref_dir = np.column_stack((temp_thetas[0,:], temp_phis))
print(pref_dir.shape)

angle_pref_x = np.sin(np.radians(pref_dir[:,1]))*np.cos(np.radians(pref_dir[:,0]))
angle_pref_y = np.sin(np.radians(pref_dir[:,1]))*np.sin(np.radians(pref_dir[:,0]))
angle_pref_z = np.cos(np.radians(pref_dir[:,1]))
fig = plt.figure(figsize = (10, 7))
ax = plt.axes(projection ="3d")
#print(angle_pref_z[0:70])
# Creating plot
ax.scatter3D(angle_pref_x,angle_pref_y,angle_pref_z)

# Z.u_i calculation
HD = np.zeros((len(pref_dir), newpos.shape[0]))
print(len(pref_dir))
for i in range(newpos.shape[0]):
  for j in range(len(pref_dir)):
    HD[j,i] = ((newpos[i,0]-newpos[0,0])*angle_pref_x[j]) + ((newpos[i,1]-newpos[0,1])*angle_pref_y[j]) + ((newpos[i,2]-newpos[0,2])*angle_pref_z[j])#incomplete PI
print(HD.shape)

#Path Integration

In [ ]:
if aligned_m == 1 or tilted_m == 1:
  beta = np.pi # scaling factor lattice mazes
if helix_m == 1:
  beta = 4 # scaling factor helical maze
if pegboard_m == 1:
  beta = 1.6 # scaling factor pegboard maze
print(HD.shape)
PI = np.transpose(np.cos(beta*(HD)))
print(PI.shape)
PI_nor = preprocessing.normalize(PI, norm='l2')
print(LA.norm(PI_nor[100,:]))
print(PI.shape)

#Model

In [ ]:
tf.keras.backend.clear_session()
input_data = Input(shape=(PI.shape[1],)) #PI input
encoder = Dense(50, input_dim = (PI.shape[1],), activation='linear')(input_data)
decoder = Dense(PI.shape[1], input_dim = (L1_neurons,), activation='linear')(encoder)
autoencoder = Model(input_data, decoder)
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
autoencoder.compile(optimizer=opt, loss='mse')
encoderModel = Model(input_data, encoder)
autoencoder.summary()

## Train model

In [ ]:
# train model stack1
PI_nor = np.array(PI_nor)
x_train, x_test, = train_test_split(PI_nor, test_size=0.2, random_state=42)


# Train the model
history = autoencoder.fit(x_train,
                x_train,
                epochs= 10,
                steps_per_epoch = 100,
                batch_size = None,
                validation_data = (x_test, x_test),
                validation_steps = 10,
                shuffle= False)
#autoencoder.save(data_fol + "autoencoder_aligned_lattice")
if aligned_m == 1:
  save_model(autoencoder, "autoencoder_AL_t"+str(n1)+"p"+str(n2+1)+"_b"+ str(beta) + '_2D')
if tilted_m == 1:
  save_model(autoencoder, "autoencoder_TL_t"+str(n1)+"p"+str(n2+1)+"_b"+ str(beta) +"_2D")
if pegboard_m == 1:
  save_model(autoencoder, "autoencoder_pegboard_t20p3_b1.6_l1")
if pegboard_m == 1:
  save_model(autoencoder, "autoencoder_helix_t20p3_b4_l1")
tf.keras.backend.clear_session()

In [ ]:
plt.plot(np.asarray(history.history['loss']),"--r", label = "loss")
plt.plot(np.asarray(history.history['val_loss']),"--b", label = "val_loss")
#plt.plot(np.asarray(hist['training_loss']), "--r", label = 'training_loss')
#plt.plot(np.asarray(hist['validation_loss']), "--b", label = 'validation_loss')
plt.legend()
plt.title("training loss")
plt.show()

#Get the output of middle layer

In [ ]:
mod_nam = "autoencoder_AL_t"+str(n1)+"p"+str(n2+1)+"_b"+ str(np.pi) +"_2D"
if aligned_m == 1:
  autoencoder = load_model(mod_nam)
  layer_output = autoencoder.get_layer("dense")
  layer_output = layer_output.output
  inp = autoencoder.input
  functor = K.function(inp, layer_output)
  encoded1 = functor([PI_nor])
if tilted_m == 1:
  autoencoder = load_model(data_fol + "autoencoder_TL_t"+str(n1)+"p"+str(n2+1)+"_b"+ str(beta) +"_2D")
  layer_output = autoencoder.get_layer("dense")
  layer_output = layer_output.output
  inp = autoencoder.input
  functor = K.function(inp, layer_output)
  encoded1 = functor([PI_nor])
# if pegboard_m == 1:
#   #layer 1
#   autoencoder = load_model(data_fol + "autoencoder_pegboard_t20p3_b1.6_l1")
#   layer_output = autoencoder.get_layer("dense")
#   layer_output = layer_output.output
#   inp = autoencoder.input
#   functor = K.function(inp, layer_output)
#   encoded1 = functor([PI_nor])
#   #layer 2
#   autoencoder = load_model(data_fol + "autoencoder_pegboard_t20p3_b1.6_l2")
#   layer_output = autoencoder.get_layer("dense")
#   layer_output = layer_output.output
#   inp = autoencoder.input
#   functor = K.function(inp, layer_output)
#   encoded2 = functor([data1])
# if helix_m == 1:
#   #layer 1
#   autoencoder = load_model(data_fol + "autoencoder_helix_t20p3_b4_l1")
#   layer_output = autoencoder.get_layer("dense")
#   layer_output = layer_output.output
#   inp = autoencoder.input
#   functor = K.function(inp, layer_output)
#   encoded1 = functor([PI_nor])
#   #layer 2
#   autoencoder = load_model(data_fol + "autoencoder_helix_t20p3_b4_l2")
#   layer_output = autoencoder.get_layer("dense")
#   layer_output = layer_output.output
#   inp = autoencoder.input
#   functor = K.function(inp, layer_output)
#   encoded2 = functor([data1])


## firing rate map func

In [ ]:
def firing_rate_map(ot, all_dat, thresh_param = 0.0, title = "", ticks_colbar = None, res_param = 40, sub_plot = False, sub_thresh=False):
  res = res_param
  if sub_thresh:
    mean_resp = np.mean(ot)
    std_resp = np.std(ot)
    thresh = mean_resp + (thresh_param * std_resp)
  else:
    thresh = np.max(ot)*thresh_param

  # H, _, _ = np.histogram2d(all_dat[:,0], all_dat[:,1], bins = res*10)

  # H[H==0] = 1
  firr = np.nonzero(ot>thresh)
  firposgrid = all_dat[firr[0], :2]
  #firr = list(firr[0])

  x = np.arange(1, 6, 1/res)
  y = np.arange(1, 6, 1/res)


  fx,fy = np.meshgrid(x, y)
  firingmap = np.zeros(fx.shape)
  firingvalue = abs(ot[firr])
  for ii in range(len(firposgrid)):
    q1 = np.argmin(abs(firposgrid[ii,0] - fx[1,:]))
    q2 = np.argmin(abs(firposgrid[ii,1] - fy[:,1]))
    firingmap[q2,q1] = max(firingvalue[ii], firingmap[q2,q1])
    # firingmap[q2,q1] += 1
  # firingmap = np.divide(firingmap, H)
  # firingmap[H==0] = 0
  # plt.imshow(firingmap)
  # plt.colorbar()
  # plt.show()

  firingmap = firingmap/max(np.max(firingmap),1)
  win_len = 9
  gaussian = matlab_style_gauss2D([win_len, win_len], 3.0)
  spikes_smooth = scipy.signal.convolve2d(gaussian, firingmap)
  rotated_img = ndimage.rotate(spikes_smooth, 1*0)

  rotated_img = rotated_img[int(win_len/2):-int(win_len/2), int(win_len/2):-int(win_len/2)]
  if not sub_plot:
    plt.imshow(rotated_img, origin= 'upper', cmap='jet')
    plt.title(title)
    plt.colorbar(ticks = ticks_colbar)
    # plt.clim(0,1)
    ax=plt.gca()                            # get the axis
    ax.set_ylim(ax.get_ylim()[::-1])        # invert the axis
    # ax.set_xlim(ax.get_xlim()[::-1])        # invert the axis
    plt.show()
    # ax.xaxis.tick_bottom()                     # and move the X-Axis
    # ax.set_yticklabels([])
    # ax.set_xticklabels([])

  return rotated_img


def matlab_style_gauss2D(shape,sigma):
  """
  2D gaussian mask - should give the same result as MATLAB's
  fspecial('gaussian',[shape],[sigma])
  """
  m,n = [(ss-1.)/2. for ss in shape]
  y,x = np.ogrid[-m:m+1,-n:n+1]
  h = np.exp( -(x*x + y*y) / (2.*sigma*sigma) )
  h[ h < np.finfo(h.dtype).eps*h.max() ] = 0
  sumh = h.sum()
  if sumh != 0:
    h /= sumh
  return h

In [ ]:
def _inf_rate(rate_map, px):
    '''A helper function for information rate.'''
    tmp_map = np.ma.array(rate_map, copy=True)
    tmp_map[np.isnan(tmp_map)] = 0
    avg_rate = np.sum(np.ravel(tmp_map * px))
    return (np.nansum(np.ravel(tmp_map * np.log2(tmp_map/avg_rate) * px)),
            avg_rate)

pos = newpos
reso = 8
H1, _, _ = np.histogram2d(pos[:,0], pos[:,1], bins=reso*5)
pos_prob = (H1)/len(pos)

## plot 2D cells

In [ ]:
pos.shape

In [ ]:

# encoded_pre = {}

# for i in range(len(lay_nam)):
#   layer_output = autoencoder_model.get_layer(lay_nam[i]).output
#   functor = K.function(inp, layer_output)
#   temp1 = functor(comp_data)
#   if lay_nam[i] == 'graph_conv':
#     encoded_pre['graph_LEC'] = temp1[:,0,:]
#     encoded_pre['graph_MEC'] = temp1[:,1,:]
#   else:
#     encoded_pre[lay_nam[i]] = temp1

firing_maps = []
spatial_info = []
pos_out = pos
lim = 2.0
from matplotlib.pyplot import close

resp_neurons = encoded1.T
for k in range(1):
  num = 0
  for j in range(int(np.divide(len(resp_neurons), resp_neurons.shape[0]))):
    for i in range(resp_neurons.shape[0]):
      axarr = plt.subplot(8,8,i+1)
      img_dat = firing_rate_map(resp_neurons[i+num], pos_out, thresh_param=lim,
                                res_param = reso, sub_plot = True, sub_thresh = True)
      axarr.imshow(img_dat, cmap = 'jet')

      # firing_maps.append(img_dat)
      sp_info_rate, avg_rate = _inf_rate(img_dat, pos_prob)
      spatial_info.append(sp_info_rate*0.1/avg_rate)
      # spatial_info[outs[k]].append(sp_info_rate/avg_rate)
      firing_maps.append(img_dat)

      plt.suptitle('output mesh with obj ' + str(num) + ' to '+str(num+resp_neurons.shape[0]) + ' neurons, threshold=' + str(lim) + " act_func=" + "relu", fontsize = 20, va = 'bottom', ha = 'center')


    num = num + resp_neurons.shape[0]
    figure = plt.gcf() # get current figure
    figure.set_size_inches(14, 10)
    # plt.savefig(fol + onm + "_t_" + str(traj1)[:-4] + "_" + str(num), bbox_inches='tight')
    plt.show()
    close()

## grid and spatial score

In [ ]:
plt.scatter(range(len(spatial_info)), spatial_info)
plt.hlines(y = 0.3, xmin=0.05, xmax=50, color='red')
plt.show()

print(np.sum(np.asarray(spatial_info) > 0.3))

In [ ]:
# from grid_score import *
# spatial_firing
scores = process_grid_data(firing_maps)
print(scores['grid_score'])


In [ ]:
key = 'grid_score'
place = (np.asarray(spatial_info) >= 0.30)*1 + (np.asarray(scores[key]) <= 0.0)*1
place_ind_temp = np.where(place == 2)[0]
place_ind = place_ind_temp
grid = (np.asarray(scores[key]) > 0.0)*1
grid_ind_temp = np.where(grid == 1)[0]
grid_ind = grid_ind_temp
# grid_ind = len(np.where(grid==2)[0])
print('place: ' + str(len(place_ind)) + 'grid: ' + str(len(grid_ind)))

In [ ]:
place_ind = [ 0,  1,  5,  7,  9, 10, 11, 13, 17, 19, 21, 22, 23, 26, 27, 28, 29, 30, 32, 33, 36, 40, 41, 45, 46, 47, 49]

In [ ]:
np.asarray(place_ind) + 1

In [ ]:
def common_elements(a, b):
  a.sort()
  b.sort()
  i, j = 0, 0
  common = []
  while i < len(a) and j < len(b):
    if a[i] == b[j]:
      common.append(a[i])
      i += 1
      j += 1
    elif a[i] < b[j]:
      i += 1
    else:
      j += 1
  return common

print('Common values:', ', '.join(map(str, common_elements(neuron_num, place_ind))))

#Save the output of the bottleneck of autoencoder for generating properties of place cells

In [ ]:
if aligned_m == 1:
  np.savetxt("encoded_AL_t"+str(n1)+"p"+str(n2+1)+"_b"+ str(3.14) +"_2D.csv", encoded1, delimiter=",")
if tilted_m == 1:
  np.savetxt(data_fol + "encoded_TL_t"+str(n1)+"p"+str(n2+1)+"_b1.8.csv", encoded1, delimiter=",")
if pegboard_m == 1:
  np.savetxt(data_fol + "encoded_pegboard_t20p3_b1.6_l1.csv", encoded1, delimiter=",")
  np.savetxt(data_fol + "encoded_pegboard_t20p3_b1.6_l2.csv", encoded2, delimiter=",")
if helix_m == 1:
  np.savetxt(data_fol + "encoded_helix_t20p3_b4_l1.csv", encoded1, delimiter=",")
  np.savetxt(data_fol + "encoded_helix_t20p3_b4_l2.csv", encoded2, delimiter=",")

#Plot the results

In [ ]:
# Layer 1 output
import plotly.offline as pyo
import plotly.io as pio
pyo.init_notebook_mode()
# place_ind = [0,  3,  4,  8, 10, 11, 12, 14, 15, 17, 18] #, 20, 23, 26, 28, 29, 30, 31, 34, 35, 38, 39, 41, 43, 45, 46, 47]
# place_ind = [ 0,  1,  2,  7, 12, 15, 19, 20, 21, 23, 27, 30, 31, 34, 35, 36, 38, 40, 42, 43, 45, 46, 47, 48, 49]
# place_ind = [3]
thresh_param= 1.3
#encoded = np.asarray(PI_nor)
encoded = np.asarray(encoded1)
if tilted_m == 1:
  traj = tilted
for neuron in place_ind: #Change here to plot different neurons response
  print(neuron)
  mean = np.mean(encoded[:,neuron])
  std = np.std(encoded[:,neuron])
  thresh = mean + thresh_param * std
  firr = np.nonzero(encoded[:,neuron]>thresh)#firr rate gives label number of firing neuron
  fig=go.Figure()
  firposgrid = traj[firr[0],:]
  fig.add_trace(go.Scatter3d(x=firposgrid[:,0],y=firposgrid[:,1],z=firposgrid[:,2],mode='markers', marker=dict(size=5)))
  fig.add_trace(go.Scatter3d(x=traj[:,0], y=traj[:,1], z=traj[:,2],mode='lines',opacity=0.1,marker=dict(size=1)))

  if pegboard_m ==1:
    fig.update_layout(
        scene = {
            "xaxis": {"nticks": 4},
            "xaxis": {"nticks": 1},
            "zaxis": {"nticks": 10},
            'camera_eye': {"x": 2, "y": 0, "z": 1.2},
            "aspectratio": {"x": 0.2, "y": 1, "z": 2}
            #"aspectratio": {"x": 1, "y": 5, "z": 10}
        })

  fig.update_layout(title=" layer 1 Neuron %d" %(neuron+1))
  pio.renderers.default = 'colab'
  fig.show()
  # fig.close()

In [ ]:
encoded[1,:]

In [ ]:
#layer 2 output
if pegboard_m == 1:
  thresh=0.5
  encoded = np.asarray(encoded2)
  for neuron in range(0,2): #Change here to plot different neurons response
      firr = np.nonzero(encoded[:,neuron]>thresh*np.amax(encoded[:,neuron]))#firr rate gives label number of firing neuron
      fig=go.Figure()
      firposgrid = traj[firr[0],:]
      fig.add_trace(go.Scatter3d(x=firposgrid[:,0],y=firposgrid[:,1],z=firposgrid[:,2],mode='markers', marker=dict(size=5)))
      fig.add_trace(go.Scatter3d(x=traj[:,0], y=traj[:,1], z=traj[:,2],mode='lines',opacity=0.1,marker=dict(size=1)))


      fig.update_layout(
          scene = {
             "xaxis": {"nticks": 4},
             "xaxis": {"nticks": 1},
             "zaxis": {"nticks": 10},
             'camera_eye': {"x": 2, "y": 0, "z": 1.2},
             "aspectratio": {"x": 0.2, "y": 1, "z": 2}

          })

      fig.update_layout(title=" layer 2 Neuron %d" %(neuron+1))

      fig.show()

if helix_m == 1:
  thresh=0.5
  encoded = np.asarray(encoded2)
  for neuron in range(1,5): #Change here to plot different neurons response
      firr = np.nonzero(encoded[:,neuron]>thresh*np.amax(encoded[:,neuron]))#firr rate gives label number of firing neuron
      fig=go.Figure()
      firposgrid = traj[firr[0],:]
      fig.add_trace(go.Scatter3d(x=firposgrid[:,0],y=firposgrid[:,1],z=firposgrid[:,2],mode='markers', marker=dict(size=5)))
      fig.add_trace(go.Scatter3d(x=traj[:,0], y=traj[:,1], z=traj[:,2],mode='lines',opacity=0.1,marker=dict(size=1)))


      #fig.update_layout(
      #    scene = {
      #       "xaxis": {"nticks": 4},
      #       "xaxis": {"nticks": 1},
      #       "zaxis": {"nticks": 10},
      #       'camera_eye': {"x": 2, "y": 0, "z": 1.2},
      #       "aspectratio": {"x": 0.2, "y": 1, "z": 2}
#
      #    })

      fig.update_layout(title=" layer 2 Neuron %d" %(neuron+1))

      fig.show()

## grid score functions

In [ ]:
import numpy as np
import pandas as pd
from skimage import measure
from scipy import misc
from scipy.ndimage import rotate
import seaborn as sns
import matplotlib.pylab as plt


'''
Shifts 2d array along given axis.
array_to_shift : 2d array that is to be shifted
n : array will be shifted by n places
axis : shift along this axis (should be 0 or 1)
'''


def shift_2d(array_to_shift, n, axis):
    shifted_array = np.zeros_like(array_to_shift)
    if axis == 0:  # shift along x axis
        if n == 0:
            return array_to_shift
        if n > 0:
            shifted_array[:, :n] = 0
            shifted_array[:, n:] = array_to_shift[:, :-n]
        else:
            shifted_array[:, n:] = 0
            shifted_array[:, :n] = array_to_shift[:, -n:]

    if axis == 1:  # shift along y axis
        if n == 0:
            return array_to_shift
        elif n > 0:
            shifted_array[-n:, :] = 0
            shifted_array[:-n, :] = array_to_shift[n:, :]
        else:
            shifted_array[:-n, :] = 0
            shifted_array[-n:, :] = array_to_shift[:n, :]
    return shifted_array


# shifts array by x and y
def get_shifted_map(firing_rate_map, x, y):
    shifted_map = shift_2d(firing_rate_map, x, 0)
    shifted_map = shift_2d(shifted_map, y, 1)
    return shifted_map


# remove from both where either of them is 0
def remove_zeros(array1, array2):
    array2 = np.nan_to_num(array2).flatten()
    array1 = np.nan_to_num(array1).flatten()
    array2_tmp = np.take(array2, np.where(array1 != 0))
    array1_tmp = np.take(array1, np.where(array2 != 0))
    array2 = np.take(array2_tmp, np.where(array2_tmp[0] != 0))
    array1 = np.take(array1_tmp, np.where(array1_tmp[0] != 0))
    return array1.flatten(), array2.flatten()


# remove from both where either of them is not a number (nan) - I am not proud of this, but nothing worked with np.nan
def remove_nans(array1, array2):
    array2 = array2.flatten()
    array2[np.isnan(array2)] = 666
    array1 = array1.flatten()
    array1[np.isnan(array1)] = 666
    array2_tmp = np.take(array2, np.where(array1 != 666))
    array1_tmp = np.take(array1, np.where(array2 != 666))
    array2 = np.take(array2_tmp, np.where(array2_tmp[0] != 666))
    array1 = np.take(array1_tmp, np.where(array1_tmp[0] != 666))
    return array1.flatten(), array2.flatten()



'''
The array is shifted along the x and y axes into every possible position where it overlaps with itself starting from
the position where the shifted array's bottom right element overlaps with the top left of the map. Correlation is
calculated for all positions and returned as a correlation_vector. The correlation vector is 2x * 2y.
'''


def get_rate_map_autocorrelogram(firing_rate_map):
    length_y = firing_rate_map.shape[0] - 1
    length_x = firing_rate_map.shape[1] - 1
    correlation_vector = np.empty((length_x * 2 + 1, length_y * 2 + 1)) * 0
    for shift_x in range(-length_x, length_x):
        for shift_y in range(-length_y, length_y):
            # shift map by x and y and remove extra bits
            shifted_map = get_shifted_map(firing_rate_map, shift_x, -shift_y)
            firing_rate_map_to_correlate, shifted_map = remove_zeros(firing_rate_map, shifted_map)

            correlation_y = shift_y + length_y
            correlation_x = shift_x + length_x

            if len(shifted_map) > 20:
                # np.corrcoef(x,y)[0][1] gives the same result for 1d vectors as matlab's corr(x,y) (Pearson)
                # https://stackoverflow.com/questions/16698811/what-is-the-difference-between-matlab-octave-corr-and-python-numpy-correlate
                correlation_vector[correlation_x, correlation_y] = np.corrcoef(firing_rate_map_to_correlate, shifted_map)[0][1]
            else:
                correlation_vector[correlation_x, correlation_y] = np.nan
    return correlation_vector


# make autocorr map binary based on threshold
def threshold_autocorrelation_map(autocorrelation_map, threshold=0.2):
    autocorrelation_map[autocorrelation_map > threshold] = 1
    autocorrelation_map[autocorrelation_map <= threshold] = 0
    return autocorrelation_map


# find peaks of autocorrelogram
def find_autocorrelogram_peaks(autocorrelation_map):
    autocorrelation_map_thresholded = threshold_autocorrelation_map(autocorrelation_map)
    autocorr_map_labels = measure.label(autocorrelation_map_thresholded)  # each field is labelled with a single digit
    field_properties = measure.regionprops(autocorr_map_labels)
    return field_properties


# calculate distances between the middle of the rate map autocorrelogram and the field centres
def find_field_distances_from_mid_point(autocorr_map, field_properties):
    distances = []
    mid_point_coord_x = np.ceil(autocorr_map.shape[0] / 2)
    mid_point_coord_y = np.ceil(autocorr_map.shape[1] / 2)

    for field in range(len(field_properties)):
        field_central_x = field_properties[field].centroid[0]
        field_central_y = field_properties[field].centroid[1]
        distance = np.sqrt(np.square(field_central_x - mid_point_coord_x) + np.square(field_central_y - mid_point_coord_y))
        distances.append(distance)
    return distances


'''
Grid spacing/wavelength:
Defined by Hafting, Fyhn, Molden, Moser, Moser (2005) as the distance from the central autocorrelogram peak to the
vertices of the inner hexagon in the autocorrelogram (the median of the six distances). This should be in cm.
'''


def calculate_grid_spacing(field_distances, bin_size):
    grid_spacing = np.median(field_distances) * bin_size
    return grid_spacing


'''
Defined by Wills, Barry, Cacucci (2012) as the square root of the area of the central peak of the autocorrelogram
divided by pi. (This should be in cm2.)
'''


def calculate_field_size(field_properties, field_distances, bin_size):
    central_field_index = np.argmin(field_distances)
    field_size_pixels = field_properties[central_field_index].area  # number of pixels in central field
    field_size = np.sqrt(field_size_pixels * np.squeeze(bin_size)) / np.pi
    return field_size


# https://stackoverflow.com/questions/481144/equation-for-testing-if-a-point-is-inside-a-circle
def in_circle(center_x, center_y, radius, x, y):
    square_dist = (center_x - x) ** 2 + (center_y - y) ** 2
    return square_dist <= radius ** 2


#  replace values not in the grid ring with nan
def remove_inside_and_outside_of_grid_ring(autocorr_map, field_properties, field_distances, inner_radius_constant=0.5, outer_radius_constant=2.5):
    ring_distances = get_ring_distances(field_distances)
    inner_radius = (np.mean(ring_distances) * inner_radius_constant) / 2
    outer_radius = (np.mean(ring_distances) * outer_radius_constant) / 2
    center_x = field_properties[np.argmin(field_distances)].centroid[0]
    center_y = field_properties[np.argmin(field_distances)].centroid[1]
    for row in range(autocorr_map.shape[0]):
        for column in range(autocorr_map.shape[1]):
            in_ring = in_circle(center_x, center_y, outer_radius, row, column)
            in_middle = in_circle(center_x, center_y, inner_radius, row, column)
            if not in_ring or in_middle:
                autocorr_map[row, column] = np.nan
    return autocorr_map


'''
Defined by Krupic, Bauza, Burton, Barry, O'Keefe (2015) as the difference between the minimum correlation coefficient
for autocorrelogram rotations of 60 and 120 degrees and the maximum correlation coefficient for autocorrelogram
rotations of 30, 90 and 150 degrees. This score can vary between -2 and 2, although generally values above
below -1.5 or above 1.5 are uncommon.
'''


def calculate_grid_score(autocorr_map, field_properties, field_distances):
    correlation_coefficients = []
    for angle in range(30, 180, 30):
        autocorr_map_to_rotate = np.nan_to_num(autocorr_map)
        rotated_map = rotate(autocorr_map_to_rotate, angle, reshape=False)  # todo fix this
        rotated_map_binary = np.round(rotated_map)
        autocorr_map_ring = remove_inside_and_outside_of_grid_ring(autocorr_map, field_properties, field_distances)
        rotated_map_ring = remove_inside_and_outside_of_grid_ring(rotated_map_binary, field_properties, field_distances)
        autocorr_map_ring_to_correlate, rotated_map_ring_to_correlate = remove_nans(autocorr_map_ring, rotated_map_ring)
        pearson_coeff = np.corrcoef(autocorr_map_ring_to_correlate, rotated_map_ring_to_correlate)[0][1]
        correlation_coefficients.append(pearson_coeff)
    grid_score = min(correlation_coefficients[i] for i in [1, 3]) - max(correlation_coefficients[i] for i in [0, 2, 4])
    return grid_score


def get_ring_distances(field_distances_from_mid_point):
    field_distances_from_mid_point = np.array(field_distances_from_mid_point)[~np.isnan(field_distances_from_mid_point)]
    ring_distances = np.sort(field_distances_from_mid_point)[1:7]
    return ring_distances


def calculate_grid_metrics(autocorr_map, field_properties):
    bin_size = 2.5  # cm
    field_distances_from_mid_point = find_field_distances_from_mid_point(autocorr_map, field_properties)
    # the field with the shortest distance is the middle and the next 6 closest are the middle 6
    ring_distances = get_ring_distances(field_distances_from_mid_point)
    grid_spacing = calculate_grid_spacing(ring_distances, bin_size)
    field_size = calculate_field_size(field_properties, field_distances_from_mid_point, bin_size)
    grid_score = calculate_grid_score(autocorr_map, field_properties, field_distances_from_mid_point)
    return grid_spacing, field_size, grid_score


def process_grid_data(firing_maps):
    rate_map_correlograms = []
    grid_spacings = []
    field_sizes = []
    grid_scores = []
    spatial_firing = {}
    for firing_map in firing_maps:
        firing_rate_map = firing_map
        rate_map_correlogram = get_rate_map_autocorrelogram(firing_rate_map)
        rate_map_correlograms.append(np.copy(rate_map_correlogram))
        field_properties = find_autocorrelogram_peaks(rate_map_correlogram)
        if len(field_properties) > 7:
            grid_spacing, field_size, grid_score = calculate_grid_metrics(rate_map_correlogram, field_properties)
            grid_spacings.append(grid_spacing)
            field_sizes.append(field_size)
            grid_scores.append(grid_score)
        else:
            print('Not enough fields to calculate grid metrics.')
            grid_spacings.append(np.nan)
            field_sizes.append(np.nan)
            grid_scores.append(0)
    spatial_firing['rate_map_autocorrelogram'] = rate_map_correlograms
    spatial_firing['grid_spacing'] = grid_spacings
    spatial_firing['field_size'] = field_sizes
    spatial_firing['grid_score'] = grid_scores
    return spatial_firing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate some random data for the heat map
x = np.random.randn(100)
y = np.random.randn(100)
z = np.random.randn(100)

# Create a figure and a 3D axis
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Create the heat map
hist, xedges, yedges = np.histogram2d(x, y, bins=10)
xpos, ypos = np.meshgrid(xedges[:-1], yedges[:-1])
zpos = np.zeros_like(xpos)
dx = dy = 0.5 * (xedges[1] - xedges[0])
dz = hist.flatten()
cmap = plt.cm.get_cmap('hot')  # Choose a color map

# Plot the 3D heat map
ax.bar3d(xpos.ravel(), ypos.ravel(), zpos.ravel(), dx, dy, dz, color=cmap(dz))

# Set labels and title
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Heat Map')

# Show the plot
plt.show()